In [0]:
create database if not exists f1_raw;

**Create Circuits Table**

In [0]:
%python
account_key = dbutils.secrets.get(scope='databricks-scope' ,key = 'databricks-strg-access-key' )

In [0]:
%python
spark.conf.set("fs.azure.account.key.databricksrg2026.dfs.core.windows.net",account_key)

In [0]:
drop table if exists f1_raw.circuit;
create table if not exists f1_raw.circuit (
  circuit_id integer,
  name string,
  location string,
  country string,
  lat double,
  lng double,
  alt integer,
  url string
)


In [0]:
COPY INTO databricks_proj.f1_raw.circuit
FROM 'abfss://raw@databricksrg2026.dfs.core.windows.net/circuits.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true',
  'mergeSchema' = 'true'
)
COPY_OPTIONS (
  'mergeSchema' = 'true'
);


In [0]:
select * from f1_raw.circuit

In [0]:
show schemas in databricks_proj

In [0]:
select current_catalog()

**create races table**

In [0]:
drop table if exists f1_raw.races;
create table if not exists f1_raw.races(
  race_year integer,
  race_name string,
  round integer,
  circuitId integer,
  name string,
  date date,
  time string,
  url string
)

In [0]:
COPY INTO databricks_proj.f1_raw.races
FROM 'abfss://raw@databricksrg2026.dfs.core.windows.net/races.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true',
  'mergeSchema' = 'true'
)
COPY_OPTIONS (
  'mergeSchema' = 'true'
);


In [0]:
select * from f1_raw.races

**Create tables for JSON files**
Constructors file
- single line json
- simple structure

In [0]:
drop table if exists f1_raw.constructors;
create table if not exists f1_raw.constructors(
constructorId integer,
constructorRef string,
name string,
nationality string,
url string
)

In [0]:
COPY INTO databricks_proj.f1_raw.constructors
FROM (
  SELECT 
    CAST(constructorId AS INT) AS constructorId,
    constructorRef,
    name,
    nationality,
    url
  FROM 'abfss://raw@databricksrg2026.dfs.core.windows.net/constructors.json'
)
FILEFORMAT = JSON;

In [0]:
COPY INTO databricks_proj.f1_raw.constructors
FROM 'abfss://raw@databricksrg2026.dfs.core.windows.net/constructors.json'
FILEFORMAT = JSON
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true',
  'mergeSchema' = 'true'
)
COPY_OPTIONS (
  'mergeSchema' = 'true'
);

In [0]:
select * from f1_raw.constructors

**Create Drivers table**
- single line json
- complex str

In [0]:
drop table if exists f1_raw.drivers;
create table if not exists f1_raw.drivers(
driverId integer,
driverRef string,
number integer,
code string,
name struct<forename:string, surname:string>,
dob date,
nationality string,
url string
)

In [0]:
COPY INTO databricks_proj.f1_raw.drivers
FROM (
  SELECT
    CAST(driverId AS INT) AS driverId,
    driverRef,
    CAST(number AS INT) AS number,
    code,
    STRUCT(name.forename, name.surname) AS name,
    TO_DATE(dob) AS dob,
    nationality,
    url
  FROM 'abfss://raw@databricksrg2026.dfs.core.windows.net/drivers.json'
)
FILEFORMAT = JSON;

In [0]:
select * from f1_raw.drivers

**Create results table**
- single line json
- simple structure

In [0]:
CREATE TABLE IF NOT EXISTS databricks_proj.f1_raw.results (
  constructorId INT,
  driverId INT,
  fastestLap STRING,
  fastestLapSpeed STRING,
  fastestLapTime STRING,
  grid INT,
  laps INT,
  milliseconds STRING,
  number STRING,
  points DOUBLE,
  position STRING,
  positionOrder INT,
  positionText STRING,
  raceId INT,
  rank STRING,
  resultId INT,
  statusId INT,
  time STRING
)
USING DELTA;

In [0]:
COPY INTO databricks_proj.f1_raw.results
FROM (
  SELECT
    CAST(constructorId AS INT) AS constructorId,
    CAST(driverId AS INT) AS driverId,
    fastestLap,
    fastestLapSpeed,
    fastestLapTime,
    CAST(grid AS INT) AS grid,
    CAST(laps AS INT) AS laps,
    milliseconds,
    number,
    CAST(points AS DOUBLE) AS points,
    position,
    CAST(positionOrder AS INT) AS positionOrder,
    positionText,
    CAST(raceId AS INT) AS raceId,
    rank,
    CAST(resultId AS INT) AS resultId,
    CAST(statusId AS INT) AS statusId,
    time
  FROM 'abfss://raw@databricksrg2026.dfs.core.windows.net/results.json'
)
FILEFORMAT = JSON;

**Create Pitstop taable**
- Multiline json
- simple strc

In [0]:
CREATE TABLE IF NOT EXISTS databricks_proj.f1_raw.pit_stops (
  driverId INT,
  lap INT,
  duration STRING,
  milliseconds STRING,
  raceId INT,
  stop INT,
  time STRING
)
USING DELTA;

In [0]:
COPY INTO databricks_proj.f1_raw.pit_stops
FROM (
  SELECT
    CAST(driverId AS INT) AS driverId,
    CAST(lap AS INT) AS lap,
    duration,
   CAST(milliseconds AS STRING) AS milliseconds,
    CAST(raceId AS INT) AS raceId,
    CAST(stop AS INT) AS stop,
    time
  FROM 'abfss://raw@databricksrg2026.dfs.core.windows.net/pit_stops.json'
)
FILEFORMAT = JSON
FORMAT_OPTIONS (
  'multiline' = 'true'
);

**Create Table for laps time**

In [0]:
drop table if exists f1_raw.laptime;
create table if not exists f1_raw.laptime (
 raceId int,
 driverId int,
 lap int,
 position int,
 time string,
 milliseconds int
)

In [0]:
COPY INTO databricks_proj.f1_raw.laptime
FROM 'abfss://raw@databricksrg2026.dfs.core.windows.net/lap_times'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true',
  'mergeSchema' = 'true'
)
COPY_OPTIONS (
  'mergeSchema' = 'true'
);

**Create  Qualifying Table**
- json
- multiline
- multiple files

In [0]:
CREATE TABLE IF NOT EXISTS databricks_proj.f1_raw.qualifying (
 constructorId INT,
 driverId INT,
 position INT,
 number INT,
 q1 STRING,
 q2 STRING,
 q3 STRING,
 qualifyingId INT,
 raceId INT

)


In [0]:
COPY INTO databricks_proj.f1_raw.qualifying
FROM (
  SELECT
    CAST(constructorId AS INT) AS constructorId,
    CAST(driverId AS INT) AS driverId,
    CAST(position AS INT) AS position,
    CAST(number AS INT) AS number,
    q1,
    q2,
    q3,
    CAST(qualifyId AS INT) AS qualifyingId,
    CAST(raceId AS INT) AS raceId
  FROM 'abfss://raw@databricksrg2026.dfs.core.windows.net/qualifying'
)
FILEFORMAT = JSON
FORMAT_OPTIONS (
  'multiline' = 'true'
);